# 06 · Análise de 5 erros reais

Selecionamos **5 erros diferentes** em `avaliacao/resultados/candidatos_erro.csv`, de tipos variados, e
para cada um registramos: **pergunta, o que o modelo fez, causa provável e correção proposta**.

Tipos de erro que o código já separa:
- `busca`: o trecho certo não veio no top-3 (o modelo nem teve chance)
- `alucinacao`: respondeu algo sem sustentação (inventou valor/telefone, ou respondeu "não consta")
- `recusa_indevida`: tinha a resposta no contexto e recusou
- `formato`: JSON inválido ou fora do esquema
- `citacao`: resposta certa, mas citou o trecho errado ou um id que não estava no contexto

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath("../src"))
import pandas as pd
from zela.base import carregar_teste, ler_jsonl

gab = {p["id"]: p for p in carregar_teste()}
corr = pd.read_csv("../avaliacao/resultados/correcao.csv")
execs = {(e["config"], e["id"]): e for c in "ABC" if os.path.exists(f"../avaliacao/resultados/execucoes_{c}.jsonl")
         for e in ler_jsonl(f"../avaliacao/resultados/execucoes_{c}.jsonl")}


def tipo_erro(l):
    e, g = execs[(l.config, l.id)], gab[l.id]
    ctx = {c["id"] for c in e["contexto"]}
    if not l.json_valido: return "formato"
    if l.alucinacao: return "alucinacao"
    if l.recusa_indevida:
        return "busca" if l.config != "A" and not (ctx & set(g["trechos_relevantes"])) else "recusa_indevida"
    if not l.deve_recusar and not l.citacao_correta: return "citacao"
    return "ok"


corr["tipo_erro"] = corr.apply(tipo_erro, axis=1)
corr[corr.tipo_erro != "ok"].groupby(["config", "tipo_erro"]).size().unstack(fill_value=0)

tipo_erro,alucinacao,citacao,formato
config,,,
A,32,4,14
B,4,2,31
C,7,0,0


In [2]:
def mostrar(config, qid):
    e, g = execs[(config, qid)], gab[qid]
    print(f"[{config}] {qid}: {g['pergunta']}")
    print("  gabarito :", g["resposta_esperada"], "| trechos:", g["trechos_relevantes"])
    print("  contexto :", e["contexto"], "| recusa por limiar:", e["recusa_por_limiar"])
    print("  saída    :", json.dumps(e["saida"], ensure_ascii=False) if e["saida"] else e["texto_bruto"][:400])
    print("  erro JSON:", e["erro_json"])

for _, l in corr[(corr.tipo_erro != "ok") & (corr.config.isin(["B", "C"]))].head(12).iterrows():
    mostrar(l.config, l.id); print()

[B] Q01: Vi um buraco grande no asfalto da Avenida Tucuruvi que pode derrubar motoqueiro. O que devo fazer?
  gabarito : Registrar o pedido de tapa-buraco o quanto antes pelos canais do SP156 (app, portal, 156 ou subprefeitura), pois o atendimento é por urgência. | trechos: ['T001', 'T002', 'T009']
  contexto : [{'id': 'T001', 'score': 0.8669}, {'id': 'T007', 'score': 0.8666}, {'id': 'T004', 'score': 0.8604}] | recusa por limiar: False
  saída    : Recuso: Não consta na base de conhecimento da ZelaCidade. Para esse assunto, consulte os canais oficiais da Prefeitura (SP156).
  erro JSON: sem_json

[B] Q02: Quanto custa solicitar o reparo de um buraco na rua?
  gabarito : É gratuito. | trechos: ['T002']
  contexto : [{'id': 'T001', 'score': 0.8744}, {'id': 'T007', 'score': 0.8728}, {'id': 'T002', 'score': 0.8635}] | recusa por limiar: False
  saída    : recusou:true
fontes:["T001","T002"]
categoria:"obstaculo_sinalizacao"
risco:"baixo"

Para obter mais detalhes sobre o custo do reparo de

## Os 5 erros escolhidos

Mesmos 5 erros da seção 06 do [`RELATORIO.md`](../RELATORIO.md).

| # | Config / pergunta | O que aconteceu | Causa | Correção proposta |
|---|---|---|---|---|
| 1 | C (API) — "Tem um buraco enorme na minha rua, o que eu faço?" *(observada na API; não está no conjunto de teste)* | Respondeu que o tapa-buraco não é feito na chuva e que recapeamento é outro pedido (T006); não disse onde pedir o conserto. A equivalente no teste (Q01) também falhou: a C copiou o T001 (definição do tapa-buraco), sem o canal. | **Busca:** top-3 do E5 = T006 (0,8666), T001 (0,8653), T007 (0,8641), separados por só 0,0025; T002 (canais) e T009 (urgência) ficaram de fora. | Re-ranking com cross-encoder ou busca híbrida E5 + TF-IDF; top-k maior quando os scores empatam. |
| 2 | B — Q03 "Uma obra da Sabesp deixou uma vala aberta na minha rua. A prefeitura vai tapar?" | Conteúdo certo (concessionária responde, citou T004), mas saiu como "Recusou: true / Fontes: [T004] / Risco: Alto…", sem JSON. É o padrão de 26 das 31 saídas inválidas da B. | **Formato:** o modelo de 1,5B sem ajuste não segue o esquema com prompt longo; imita a lista de campos e a regra de recusa do prompt. | LoRA (C: 100% JSON válido); sem treino: decodificação restrita por esquema JSON (gramática). |
| 3 | A — Q09 "Qual é o número de WhatsApp da prefeitura de SP para pedir tapa-buraco?" | Inventou "(+55) 11 3289-7000" da "Secretaria Municipal de Obras" e citou T001 sem ter recebido trecho nenhum. Certo: (11) 3230-5156 (T007). | **Alucinação sem contexto:** o modelo preenche a lacuna com um número plausível. | Não servir a config A (B e C deram o número certo); validar telefones da resposta contra os trechos citados. |
| 4 | C — Q41 "Furei o pneu num buraco da rua. Como peço indenização da prefeitura?" ("não consta") | Respondeu com o texto do T004 (buraco aberto por concessionária). | **Recusa:** pergunta do mesmo tema passa no limiar (0,8582 > 0,8511); o SFT teve 25 recusas contra 94 respostas, a maioria fora do tema, e o LoRA aprendeu a copiar o trecho mais próximo. | Mais negativos difíceis no SFT: perguntas do tema sem resposta no contexto (indenização, prazos, valores). |
| 5 | C — Q27 "A luz do poste em frente de casa fica acesa durante a tarde toda. Devo reportar?" | Copiou o T029 (rua sem luz × casa sem energia), embora o T028 — que responde exatamente ("acesa nos períodos da manhã e da tarde") — fosse o 1º do contexto. | **Escolha do trecho / cópia literal:** o alvo do SFT era o texto inteiro de um trecho; o LoRA aprendeu a copiar um trecho em vez de responder à pergunta, e às vezes escolhe o errado. | Alvos do SFT redigidos como resposta à pergunta, com o trecho certo em posições variadas do contexto. |